In [1]:
# 노이즈 첨가할 때 전처리 유뮤에 따른 성능 감소 
import os
import torch
import cv2
from torchvision import datasets, transforms
from PIL import Image
import numpy as np
import random 

class preprocessing_baseline:
    def __init__(self, threshold=128):
        self.threshold = threshold
    
    def __call__(self, img):
        img_np = np.array(img.convert("RGB"))
        resized = cv2.resize(img_np, (64,64))
        gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
        _, binarized = cv2.threshold(gray, self.threshold, 255, cv2.THRESH_BINARY)
        return binarized

class proprocessing_ours:
    def __init__(self, size, threshold, crop, method):
        self.size = size
        self.threshold = threshold
        self.crop = crop
        self.method = method  # 'Fixed', 'Otsu', 'Adaptive'

    def __call__(self, img):
        # 1. PIL → NumPy (RGB) & Crop
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w,_ = img_np.shape
            img_np = img_np[self.crop:h - self.crop, self.crop:w - self.crop, :]

        # 2. Grayscale
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        # 3. Edge extraction(LoG)
        log_edge = np.abs(cv2.Laplacian(gray, cv2.CV_64F, ksize=3)) 
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3);sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_edge = np.sqrt(sobelx ** 2 + sobely ** 2)

        # 먼저 정규화 (0~1)
        log_norm = cv2.normalize(log_edge, None, 0, 1.0, cv2.NORM_MINMAX)
        sobel_norm = cv2.normalize(sobel_edge, None, 0, 1.0, cv2.NORM_MINMAX)
        # 자동 가중치: 평균 강도에 비례해서 반영 (더 강한 에지에 더 가중치)
        log_weight = np.mean(log_norm)
        sobel_weight = np.mean(sobel_norm)
        total = log_weight + sobel_weight
        w1 = log_weight / total
        w2 = sobel_weight / total
        # 가중 평균
        edge = w1 * log_norm + w2 * sobel_norm
        edge = cv2.normalize(edge, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8) 
 
        # 4. Gaussian Blur & Resize( resize 필수 (프로젝트 가이드라인)) 
        blurred = cv2.GaussianBlur(edge, (3, 3), 1)
        resized = cv2.resize(blurred, (self.size, self.size), interpolation=cv2.INTER_CUBIC)
        resized = resized.astype(np.uint8) 
        
        # 5. Binarization by method
        if self.method == 'Fixed':
            _, binarized = cv2.threshold(resized, self.threshold, 255, cv2.THRESH_BINARY)
        elif self.method == 'Otsu':
            _, binarized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif self.method == 'Adaptive':
            binarized = cv2.adaptiveThreshold(resized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                              cv2.THRESH_BINARY, 11, 2)
        else:
            raise ValueError(f"Unknown binarization method: {self.method}")
        
        return binarized

class AddRandomNoise:
    def __init__(self, candidate_ratios=[0.05, 0.10, 0.25, 0.50]):
        self.candidate_ratios = candidate_ratios

    def __call__(self, img):
        assert isinstance(img, np.ndarray)  

        img_np = img.copy()
        h, w = img_np.shape
        num_pixels = h * w

        # 후보 중에서 무작위로 하나 선택
        noise_ratio = random.choice(self.candidate_ratios)
        num_noisy = int(num_pixels * noise_ratio)

        if num_noisy == 0:
            return Image.fromarray(img_np.astype(np.uint8))

        coords = np.random.choice(num_pixels, num_noisy, replace=False)
        y, x = np.unravel_index(coords, (h, w))
        img_np[y, x] = 255 - img_np[y, x]

        return Image.fromarray(img_np.astype(np.uint8))

class AddNoise:
    def __init__(self, noise_ratio):
        self.noise_ratio = noise_ratio

    def __call__(self, img):
        assert isinstance(img, np.ndarray) 
        img_np = img.copy()
        h, w = img_np.shape
        num_pixels = h * w
        num_noisy = int(num_pixels * self.noise_ratio)
        coords = np.random.choice(num_pixels, num_noisy, replace=False)
        y, x = np.unravel_index(coords, (h, w))
        img_np[y, x] = 255 - img_np[y, x]
        return Image.fromarray(img_np.astype(np.uint8))

In [45]:
size_transform = transforms.Compose([
    preprocessing_baseline(threshold=128),
    AddRandomNoise(),
    transforms.ToTensor(),
])

trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train', transform=size_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True) 

# of trainset = 8000
# of trainloader = 200
input size: torch.Size([1, 64, 64]), 2


In [2]:
import numpy as np 
import torch.optim as optim   
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs, save_path='./prob2_1_pre_weight'):
    os.makedirs(save_path, exist_ok=True)  
    history = np.zeros((0, 3))  

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss, correct, total = 0, 0, 0

        for X, y in trainloader: 
            X = X.to(device);y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total
        history = np.vstack((history, [epoch + 1, avg_loss, avg_accuracy]))
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

        if (epoch + 1) % 10 == 0:
            filename = os.path.join(save_path, f"prob2_1_1_weight{epoch+1}.pth")
            torch.save(model.state_dict(), filename)
            print(f"Saved checkpoint: {filename}")
    return history
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [47]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
prob2_1 = models.resnet18()
num_ftrs = prob2_1.fc.in_features
prob2_1.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
prob2_1.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)

prob2_1 = prob2_1.to(device)

# 하이퍼파라미터 설정
num_epochs = 50
lr = 0.001
optimizer = torch.optim.Adam(prob2_1.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(prob2_1, device, trainloader, optimizer, criterion, num_epochs)

  2%|▉                                           | 1/50 [00:12<10:04, 12.33s/it]

Epoch [1/50] - Loss: 1.344921, Accuracy: 0.3485


  4%|█▊                                          | 2/50 [00:24<09:51, 12.33s/it]

Epoch [2/50] - Loss: 1.226658, Accuracy: 0.4273


  6%|██▋                                         | 3/50 [00:37<09:39, 12.34s/it]

...

Epoch [48/50] - Loss: 0.437696, Accuracy: 0.7771


 98%|██████████████████████████████████████████▏| 49/50 [10:07<00:12, 12.41s/it]

Epoch [49/50] - Loss: 0.416455, Accuracy: 0.7873


100%|███████████████████████████████████████████| 50/50 [10:19<00:00, 12.39s/it]

Epoch [50/50] - Loss: 0.402958, Accuracy: 0.7930
✅ Saved checkpoint: ./checkpoints5/prob2_1_1_weight50.pth


In [5]:
import torch
import os
import torchvision.models as models
import torch
import torch.nn as nn

test_transform = transforms.Compose([
    Binarize(threshold=128),
    AddNoise(noise_ratio=0.05), #0.05, 0.10, 0.25, 0.50
    transforms.ToTensor(),
])
testset  = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform=test_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size = 40, shuffle=True, num_workers=2, drop_last=True)
model_paths = [
    './prob2_1_pre_weight/prob2_1_1_weight10.pth',
    './prob2_1_pre_weight/prob2_1_1_weight20.pth',
    './prob2_1_pre_weight/prob2_1_1_weight30.pth',
    './prob2_1_pre_weight/prob2_1_1_weight40.pth',
    './prob2_1_pre_weight/prob2_1_1_weight50.pth',
]
criterion = torch.nn.CrossEntropyLoss()
for path in model_paths:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = models.resnet18()
    num_ftrs = model.fc.in_features
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4)  
)
    model.load_state_dict(torch.load(path))
    model = model.to(device)
    test_result = test(model, device, testloader, criterion)

100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 18.43it/s]


test loss : 1.0481 / test_accuracy : 0.5526


100%|███████████████████████████████████████████| 19/19 [00:00<00:00, 20.76it/s]


test loss : 1.3435 / test_accuracy : 0.6382


100%|███████████████████████████████████████████| 19/19 [00:00<00:00, 20.40it/s]


test loss : 1.6974 / test_accuracy : 0.6566


100%|███████████████████████████████████████████| 19/19 [00:00<00:00, 20.60it/s]


test loss : 2.1602 / test_accuracy : 0.6395


100%|███████████████████████████████████████████| 19/19 [00:00<00:00, 20.05it/s]

test loss : 2.0119 / test_accuracy : 0.6342
